# Delay-FC Demo

How per-region haemodynamic delays alter functional connectivity.

In [ ]:
import numpy as np
import sys; sys.path.insert(0, "..")
from src.neural_generator import generate_coupled_oscillators
from src.bw_model import BWParams, simulate, downsample
from src.fc_from_bold import fc_legacy, fc_delay_corrected, fc_difference, pearson_r_fc
params = BWParams()
dt, T, tr = 0.001, 120.0, 2.0
C2 = np.array([[0.0, 0.3], [0.3, 0.0]])
neural_2 = generate_coupled_oscillators(2, T, dt, C2, np.array([0.05, 0.05]))
print("Neural shape:", neural_2.shape)

In [ ]:
r1 = simulate(neural_2[0], dt, T, params).bold
r2 = simulate(neural_2[1], dt, T, params).bold
bd = np.stack([downsample(r1, dt, tr), downsample(r2, dt, tr)])
fc_nd = fc_legacy(bd, tr)
print("FC no delay:", fc_nd[0,1])

In [ ]:
from src.delay_inject import inject_delays_bw
d2s = np.array([0.0, 2.0])
bdel = inject_delays_bw(neural_2, d2s, params, dt, T)
bddl = np.stack([downsample(bdel[i], dt, tr) for i in range(2)])
fcwd = fc_legacy(bddl, tr)
fcc = fc_delay_corrected(bddl, d2s, tr)
print("FC uncorrected:", fcwd[0,1])
print("FC corrected:", fcc[0,1])

In [ ]:
dfc = fc_difference(fcwd, fcc)
print("DeltaFC[0,1]:", dfc[0,1])

In [ ]:
n = 20
dst = np.abs(np.subtract.outer(np.arange(n), np.arange(n))).astype(float)
C20 = np.exp(-dst / 5.0); np.fill_diagonal(C20, 0)
d20 = np.concatenate([np.zeros(10), np.linspace(0.5, 2.0, 10)])
n20 = generate_coupled_oscillators(n, T, dt, C20, np.full(n, 0.05))
b20 = inject_delays_bw(n20, d20, params, dt, T)
b20d = np.stack([downsample(b20[i], dt, tr) for i in range(n)])
delta20 = fc_difference(fc_legacy(b20d, tr), fc_delay_corrected(b20d, d20, tr))
print("Mean abs delta FC:", abs(delta20[~np.eye(n, dtype=bool)]).mean())

## Why this matters for TVB

Delays bias empirical FC. Delay correction removes this confound.